## Lab 03 
- Francisco Javier De la Torre Silva 745974

  
18/02/2026

In [24]:
from spark_utils import SparkUtils

In [25]:
spark_url = "spark://spark-master:7077"
app_name = "Lab03"
su = SparkUtils(spark_url, app_name)
su

<SparkContext master=spark://spark-master:7077 appName=Lab03>

In [26]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,
Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
column_types = [
    ("Order_ID","integer"),
    ("Company","string"),
    ("City","string"),
    ("Customer_Age","integer"),
    ("Order_Value","float"),
    ("Delivery_Time_Min","float"),
    ("Distance_Km","float"),
    ("Items_Count","float"),
    ("Product_Category","string"),
    ("Payment_Method","string"),
    ("Customer_Rating","float"),
    ("Discount_Applied","float"),
    ("Delivery_Partner_Rating","float"),
]

ecommerce_schema = SparkUtils.generate_schema(column_types)

In [27]:
ecommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(ecommerce_schema) \
                .csv("/opt/spark/work-dir/data/ecommerce/")
ecommerce_df.printSchema()
print(f"rows number: {ecommerce_df.count()}")
ecommerce_df.show(5)

root
 |-- Order_ID: integer (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

rows number: 1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+------

In [28]:
from pyspark.sql.functions import when, count, isnull

ecommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [29]:
ecommerce_null_count_df = ecommerce_df.select([count(when(isnull(c), c)).alias(c) for c in ecommerce_df.columns])
ecommerce_null_count_df.show() 

[Stage 38:=============================>                            (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [30]:
ecommerce_null_count_df_2 = SparkUtils.count_nulls(ecommerce_df)
ecommerce_null_count_df_2.show()

[Stage 41:=============================>                            (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



Drop nulls

In [31]:
ecommerce_wo_nulls = ecommerce_df.dropna()
print(f"size of df after removing nulls: {ecommerce_wo_nulls.count()}")

[Stage 44:=============================>                            (1 + 1) / 2]

size of df after removing nulls: 780992


fill null values

In [32]:
ecommerce_wo_nulls_fillna = ecommerce_df.fillna({
    'City': 'Unknown',
    'Items_Count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print(f"number of nulls after removing nulls: {ecommerce_wo_nulls_fillna.count()}")

number of nulls after removing nulls: 1000000


In [45]:
from pyspark.sql.functions import lit, col, avg, count
ecommerce_wo_nulls_fillna = ecommerce_wo_nulls_fillna.withColumn("Code_Country", lit("IN"))
ecommerce_wo_nulls_fillna.show(1)

+--------+----------------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|        Wallet|            2.1|             1.0|                    3.2|          IN|
+--------+----------------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------

Add tax column

In [41]:
ecommerce_tax_df = ecommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)

### Task 1: Delivery SLA classification + filtering

In [42]:
ecommerce_df_task1 = ecommerce_tax_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST") \
                                                                 .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME") \
                                                                 .when(col("Delivery_Time_Min") > 20, "LATE")) \
                                                                 .filter(col("Delivery_SLA") == "LATE") \
                                                                 .orderBy(col("Delivery_Time_Min"), ascending=False)

ecommerce_df_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

### Task 2: Discount impact (effective price + bucket + grouping)

In [44]:
ecommerce_df_task2 = ecommerce_df_task1.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))) \
                                        .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
                                                                    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
                                                                    .when(col("Effective_Order_Value") > 500, "HIGH"))

ecommerce_df_task2.groupBy("Price_Bucket") \
    .agg(count("*").alias("Count"), avg("Effective_Order_Value").alias("Avg_Effective_Order_Value")) \
    .orderBy(col("Avg_Effective_Order_Value").desc()) \
    .show()

[Stage 63:=============================>                            (1 + 1) / 2]

+------------+------+-------------------------+
|Price_Bucket| Count|Avg_Effective_Order_Value|
+------------+------+-------------------------+
|        HIGH| 56013|         707.219837039914|
|      MEDIUM| 59824|       353.49134017350553|
|         LOW|143811|        25.36497016341368|
+------------+------+-------------------------+



### Task 3:  Customer segmentation (Age_Group) + category insights

In [46]:
ecommerce_df_task3 = ecommerce_df_task2.filter(col("Customer_Age") \
                                       .isNotNull() & (col("Customer_Age") >= 0) & (col("Customer_Age") <= 100)) \
                                       .withColumn("Age_Group", when(col("Customer_Age") < 25, "YOUNG") \
                                       .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT") \
                                       .when(col("Customer_Age") >= 45, "SENIOR"))

ecommerce_df_task3.groupBy("Age_Group", "Product_Category") \
                  .agg(count("*").alias("orders"), avg("Order_Value").alias("avg_order_value")) \
                  .orderBy(col("Age_Group").asc(), col("orders").desc()) \
                  .show()

[Stage 71:=============================>                            (1 + 1) / 2]

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|orders|   avg_order_value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17935|495.73422188930334|
|    ADULT|              Dairy| 17780| 502.4952578519943|
|    ADULT|          Household| 17767|497.23453261697006|
|    ADULT|             Snacks| 17700|499.79997128481244|
|    ADULT|Fruits & Vegetables| 17671|494.20056196792063|
|    ADULT|          Beverages| 17650| 496.7228836926252|
|    ADULT|          Groceries| 17545|491.13630949723205|
|   SENIOR|          Groceries| 13440|500.87560482649576|
|   SENIOR|      Personal Care| 13253| 501.4177858687559|
|   SENIOR|             Snacks| 13250|502.81940932795686|
|   SENIOR|              Dairy| 13176|498.58672649583895|
|   SENIOR|          Household| 13163|  489.620170382284|
|   SENIOR|          Beverages| 13118|492.35374550340305|
|   SENIOR|Fruits & Vegetables| 12955|498.62873171412866|
|    YOUNG|   